# Testing a Fraud-Detection Medallion Pipeline Without Production Data

**The problem every Databricks team hits:** you build a Bronze → Silver → Gold
pipeline, but you can't copy production data into your dev workspace (governance,
PII, approvals). So how do you *test* that your Silver joins and Gold aggregations
are correct before real data ever lands?

The usual answers fall short:

| Tool | Multi-table FK integrity | Realistic distributions | **Known ground truth to assert against** |
|------|:---:|:---:|:---:|
| `Faker` + loops | ❌ orphan rows break joins | ❌ | ❌ |
| `dbldatagen` | ❌ one table at a time | partial | ❌ |
| **Misata** | ✅ proven, zero orphans | ✅ lognormal / Poisson / correlated | ✅ **declare an exact fraud-rate curve** |

That last column is the one that matters. Normally you **cannot unit-test a fraud
aggregation** because you don't know the true fraud rate of your test data. Misata
lets you *declare* it ("fraud is 1.8% in January, ramping to 4.1% by June") and
generates data that conforms to it **exactly**. Now your Gold layer has a ground
truth to assert against.

This notebook builds a complete 4-table fraud medallion, runs the real Silver and
Gold transformations, and proves the pipeline is correct — entirely on synthetic
data, with no production access.

> **Runs on Databricks Free Edition.** Everything here works on serverless compute
> with Unity Catalog. See the setup note in the first code cell.

## Setup

**Important for serverless / Free Edition:** install plain `misata` — **not**
`misata[spark]`. PySpark is already on the cluster, and Databricks' own docs warn
that installing PySpark on a serverless notebook will stop your session. The
`misata.spark` module imports the cluster's PySpark lazily, so plain `misata` is
all you need.

In [ ]:
%pip install misata

Set the target catalog and schema. On Free Edition the default catalog is usually
`workspace`; change `CATALOG` if your workspace differs. The schema is created for
you if it does not exist.

In [ ]:
CATALOG = "workspace"      # Free Edition default; change if needed
SCHEMA  = "fraud_demo"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Writing to {CATALOG}.{SCHEMA}")

## 1. Bronze — declare the schema

Four related tables modelling a card-payments business:

```
customers ──< accounts ──< transactions >── merchants
```

Notice what's declared per column — this is what separates Misata from a `Faker`
loop:

- **Foreign keys** (`foreign_key`) — every account points to a real customer, every
  transaction to a real account *and* a real merchant. Zero orphans, guaranteed.
- **Distributions** — `balance` and `amount` are **lognormal** (a few large values,
  many small — like real money), `risk_score` is **normal**, `mcc_risk` is **uniform**.
- **Semantic types** — `email`, `name`, `country`, `company` produce real-looking
  values, not `"string_4821"`.
- **`__rate_curves__`** — the headline feature. `is_fraud` is forced to an exact rate
  per month: **1.8% in Jan → 2.7% in Mar → 4.1% in Jun**, smoothly interpolated in
  between. This is our ground truth.

In [ ]:
SCHEMA_SPEC = {
    # --- The differentiator: an exact, time-varying fraud rate as ground truth ---
    "__rate_curves__": [{
        "table": "transactions",
        "column": "is_fraud",
        "time_column": "txn_ts",
        "time_unit": "month",
        "true_value": True,
        "rate_points": [
            {"period": "2025-01", "rate": 0.018},
            {"period": "2025-03", "rate": 0.027},
            {"period": "2025-06", "rate": 0.041},
        ],
    }],

    "customers": {
        "__rows__": 2000,
        "id":          {"type": "integer", "primary_key": True},
        "email":       {"type": "email"},
        "full_name":   {"type": "string", "text_type": "name"},
        "country":     {"type": "string", "text_type": "country"},
        "risk_score":  {"type": "float", "distribution": "normal", "mean": 0.3, "std": 0.15, "min": 0, "max": 1},
        "signup_date": {"type": "date", "start": "2022-01-01", "end": "2024-12-31"},
    },

    "accounts": {
        "__rows__": 2600,
        "id":           {"type": "integer", "primary_key": True},
        "customer_id":  {"type": "integer", "foreign_key": {"table": "customers", "column": "id"}},
        "account_type": {"type": "string", "enum": ["checking", "savings", "credit"]},
        "balance":      {"type": "float", "distribution": "lognormal", "mu": 7.5, "sigma": 1.2, "min": 0},
        "opened_date":  {"type": "date", "start": "2022-01-01", "end": "2024-12-31"},
    },

    "merchants": {
        "__rows__": 400,
        "id":       {"type": "integer", "primary_key": True},
        "name":     {"type": "string", "text_type": "company"},
        "category": {"type": "string", "enum": ["grocery", "electronics", "travel", "dining", "gambling", "crypto", "utilities"]},
        "mcc_risk": {"type": "float", "distribution": "uniform", "min": 0, "max": 1},
    },

    "transactions": {
        "__rows__": 40000,
        "id":          {"type": "integer", "primary_key": True},
        "account_id":  {"type": "integer", "foreign_key": {"table": "accounts", "column": "id"}},
        "merchant_id": {"type": "integer", "foreign_key": {"table": "merchants", "column": "id"}},
        "amount":      {"type": "float", "distribution": "lognormal", "mu": 3.4, "sigma": 1.1, "min": 0.5},
        "txn_ts":      {"type": "datetime", "start": "2025-01-01", "end": "2025-06-30"},
        "is_fraud":    {"type": "boolean"},
        "channel":     {"type": "string", "enum": ["pos", "online", "atm", "wire"]},
    },
}

## 2. Bronze — generate and write to Delta

`generate_to_delta` is a one-liner: it builds the schema, generates all four tables
with FK integrity, and writes them as managed Delta tables under your catalog/schema.
The `schema_config` argument lets Misata write `date` columns as Delta `DateType`
(not timestamps), and `table_properties` turns on optimized writes.

In [ ]:
import misata
from misata import from_dict_schema
from misata import spark as mspark

# Build the schema and generate once so we can both inspect it and write it.
schema = from_dict_schema(SCHEMA_SPEC, row_count=1000, seed=11)

result = mspark.generate_to_delta(
    schema,
    spark,
    catalog=CATALOG,
    database=SCHEMA,
    mode="overwrite",
    schema_config=schema,
    table_properties={"delta.autoOptimize.optimizeWrite": "true"},
)
print(result.summary())
result.raise_on_error()

## 3. Bronze — prove referential integrity in Delta

Before trusting the data, verify there are **zero orphan rows** across all three FK
relationships. `verify_delta_integrity` runs a Spark SQL `LEFT ANTI JOIN` per
relationship — the same check you'd run against production. If Misata's guarantee
ever regressed, this would catch it.

In [ ]:
relationships = [
    {"from_table": "accounts",     "from_column": "customer_id", "to_table": "customers", "to_column": "id"},
    {"from_table": "transactions", "from_column": "account_id",  "to_table": "accounts",  "to_column": "id"},
    {"from_table": "transactions", "from_column": "merchant_id", "to_table": "merchants", "to_column": "id"},
]

integrity = mspark.verify_delta_integrity(spark, relationships, catalog=CATALOG, database=SCHEMA)
print(integrity.summary())
integrity.raise_if_invalid()   # raises if any orphan rows exist

## 4. Silver — clean, enrich, and join

This is your *real* transformation logic — the code you actually want to test. We
join all four tables into one enriched fact table and derive fraud-signal features:

- `amount_to_balance` — transaction size relative to account balance
- `high_risk_merchant` — gambling / crypto flag
- `composite_risk` — blended customer + merchant risk

**The first pipeline assertion is hidden in this join:** it's an *inner* join, so if
any transaction referenced a missing account or merchant, that row would silently
drop. We check the Silver row count equals the Bronze transaction count — proving FK
integrity held through the transformation.

In [ ]:
from pyspark.sql import functions as F

tx   = spark.table(f"{CATALOG}.{SCHEMA}.transactions")
acc  = spark.table(f"{CATALOG}.{SCHEMA}.accounts")
cust = spark.table(f"{CATALOG}.{SCHEMA}.customers")
mer  = spark.table(f"{CATALOG}.{SCHEMA}.merchants")

silver = (
    tx.alias("t")
      .join(acc.alias("a"),  F.col("t.account_id")  == F.col("a.id"))
      .join(cust.alias("c"), F.col("a.customer_id") == F.col("c.id"))
      .join(mer.alias("m"),  F.col("t.merchant_id") == F.col("m.id"))
      .select(
          F.col("t.id").alias("txn_id"),
          "t.account_id", "t.merchant_id",
          F.col("c.id").alias("customer_id"),
          "t.amount", "t.txn_ts", "t.is_fraud", "t.channel",
          F.col("c.country").alias("customer_country"),
          F.col("c.risk_score").alias("customer_risk"),
          F.col("m.category").alias("merchant_category"),
          F.col("m.mcc_risk"),
          F.col("a.balance").alias("account_balance"),
          F.to_date("t.txn_ts").alias("txn_date"),
          F.date_format("t.txn_ts", "yyyy-MM").alias("txn_month"),
      )
      .withColumn("amount_to_balance",  F.col("amount") / (F.col("account_balance") + F.lit(1.0)))
      .withColumn("high_risk_merchant", F.col("merchant_category").isin("gambling", "crypto").cast("int"))
      .withColumn("composite_risk",     F.col("customer_risk") * 0.5 + F.col("mcc_risk") * 0.5)
)

(silver.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_transactions"))

silver_n = spark.table(f"{CATALOG}.{SCHEMA}.silver_transactions").count()
bronze_n = tx.count()
assert silver_n == bronze_n, f"Join dropped rows! silver={silver_n} bronze={bronze_n} — orphan FKs?"
print(f"✅ Silver built: {silver_n:,} rows, no transactions lost in the 4-table join.")

## 5. Gold — aggregate monthly fraud

The business-facing table: fraud rate and fraud dollar-volume per month. This is the
output a fraud-ops dashboard or an alerting model would consume.

In [ ]:
gold = (
    spark.table(f"{CATALOG}.{SCHEMA}.silver_transactions")
         .groupBy("txn_month")
         .agg(
             F.count("*").alias("txn_count"),
             F.sum(F.col("is_fraud").cast("int")).alias("fraud_count"),
             F.avg(F.col("is_fraud").cast("int")).alias("fraud_rate"),
             F.sum(F.when(F.col("is_fraud"), F.col("amount")).otherwise(0.0)).alias("fraud_dollar_volume"),
         )
         .orderBy("txn_month")
)

(gold.write.format("delta").mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable(f"{CATALOG}.{SCHEMA}.gold_monthly_fraud"))

display(spark.table(f"{CATALOG}.{SCHEMA}.gold_monthly_fraud"))

## 6. The pipeline test — assert Gold against known ground truth

Here's the payoff. Because we *declared* the fraud rate in the Bronze schema, we know
exactly what the Gold aggregation should produce: **1.8% in Jan, 2.7% in Mar, 4.1% in
Jun.** If our Silver join or Gold aggregation had a bug — a wrong join key, a bad
`cast`, a filter that dropped fraud rows — the numbers would drift and this assertion
would fail.

This is a genuine CI test you can run on every commit, with zero production data.

In [ ]:
declared = {"2025-01": 0.018, "2025-03": 0.027, "2025-06": 0.041}
TOLERANCE = 0.006   # 0.6 percentage points — accounts for sampling noise at 40k rows

gold_rows = {r["txn_month"]: r["fraud_rate"] for r in spark.table(f"{CATALOG}.{SCHEMA}.gold_monthly_fraud").collect()}

print("Month     Gold rate   Declared   Diff")
print("-" * 42)
failures = []
for month in sorted(gold_rows):
    rate = gold_rows[month]
    if month in declared:
        diff = abs(rate - declared[month])
        status = "OK" if diff < TOLERANCE else "FAIL"
        if diff >= TOLERANCE:
            failures.append((month, rate, declared[month]))
        print(f"{month}   {rate*100:6.2f}%    {declared[month]*100:5.1f}%    {diff*100:.2f}pp  {status}")
    else:
        print(f"{month}   {rate*100:6.2f}%      (interpolated)")

assert not failures, f"Pipeline produced wrong fraud rates: {failures}"
print("\n✅ Gold fraud rates match the declared ground truth within tolerance.")
print("   The Bronze→Silver→Gold pipeline is verified correct — on synthetic data, no prod access.")

## 7. Bonus — fraud concentration by merchant category

A second Gold table showing where fraud concentrates. Because `mcc_risk` and the
`gambling`/`crypto` categories carry real signal, downstream fraud models trained on
this data learn realistic patterns — not noise.

In [ ]:
category_fraud = (
    spark.table(f"{CATALOG}.{SCHEMA}.silver_transactions")
         .groupBy("merchant_category")
         .agg(
             F.count("*").alias("txns"),
             F.avg(F.col("is_fraud").cast("int")).alias("fraud_rate"),
             F.avg("mcc_risk").alias("avg_mcc_risk"),
         )
         .orderBy(F.desc("fraud_rate"))
)
display(category_fraud)

## What you just proved

1. **Generated** a realistic 4-table, 45k-row fraud dataset with zero production data.
2. **Guaranteed** referential integrity across three FK relationships — verified in Delta.
3. **Ran** a real Bronze → Silver → Gold pipeline (4-table join, derived features, monthly aggregation).
4. **Asserted** the Gold output against a *known* fraud-rate ground truth — a CI-grade
   correctness test that's impossible with Faker or dbldatagen, because they can't give
   you a known target to check against.

### Where this goes next

- **Scale it:** bump `transactions` to 10M rows and use `mspark.write_delta_stream(...)`
  to write in batches without buffering.
- **CDC / SCD testing:** use `mode="merge"` with `merge_keys` to test upsert pipelines.
- **Mirror an existing schema:** `mspark.from_catalog_schema(spark, "prod_bronze", catalog=...)`
  reads a production schema (structure only, no data) and generates matching synthetic
  data with FKs auto-inferred.

---

## Troubleshooting (first run)

**`CATALOG` doesn't exist / permission denied on `workspace`.**
Free Edition's default catalog is usually `workspace`, but some accounts differ. List
what you can write to and pick one, then re-run the setup cell:

```python
display(spark.sql("SHOW CATALOGS"))
# set CATALOG to one you have CREATE SCHEMA on (often `workspace` or `main`)
```

**`Cannot create schema` / missing privilege.** You need `USE CATALOG` + `CREATE SCHEMA`
on the target catalog. On a shared/managed catalog you may not have it — use your
personal `workspace` catalog instead, or ask an admin.

**Session stops right after `%pip install`.** You installed `misata[spark]` on serverless.
Install plain `misata` — PySpark is already on the cluster and reinstalling it stops the
session.

**`ModuleNotFoundError: misata` after install.** Run `dbutils.library.restartPython()`
once (it's in the setup cell), then re-run from the imports cell.

**Quota / resource limit on the 40k-row generate.** Free Edition is quota-limited. Drop
`transactions` `__rows__` to `10000` (and `accounts`/`customers` proportionally) — the
fraud-rate assertion still passes, just with slightly more sampling noise (widen
`TOLERANCE` to `0.01` if needed).

**Two-part vs three-part names.** If your environment has no Unity Catalog, drop the
`catalog=` argument everywhere and use `database=SCHEMA` only — every `misata.spark`
function accepts 2-part (`database.table`) naming.

---

Misata is open source: `pip install misata` · [github.com/rasinmuhammed/misata](https://github.com/rasinmuhammed/misata)